# Understanding Hash

**Session 05 (2026-10-17) · Assignment 1 of 3**

A **hash** turns any piece of data into a number. That single idea is what makes Python's `dict` and `set` so fast: instead of searching through every item, Python calculates where an item *should* be and goes straight there.

What you'll learn:

1. What a hash function is, and what makes a good one
2. Python's built-in `hash()` and its rules
3. Hashable vs unhashable types, and why a list can't be a dict key
4. How a hash table (`dict` / `set`) uses hashes to store and find data
5. Collisions, and the two ways to handle them
6. Load factor and resizing
7. Our own hash table, built from scratch in [solution.py](solution.py)
8. Why lookups are fast: list vs set vs dict
9. Cryptographic hashes with `hashlib` (SHA-256), and where they are used
10. Practice exercises, with solutions

> Run the cells from top to bottom. Examples that fail on purpose are wrapped in `try` / `except`, so **Run All** works. Some outputs (string hashes, timings, memory sizes) change from run to run; the notes explain why.

---
## 1. What is a hash?

**What it is:** a **hash function** takes an input of any size (a number, a word, a whole file) and returns a fixed-size number called its **hash** (also called a hash value or digest).

Think of it as a **fingerprint** for data: small, quick to compare, and the same data always gives the same fingerprint.

| Property | Meaning |
|---|---|
| Deterministic | The same input always gives the same hash |
| Fixed size | Short or long, every input gives an output of the same size |
| Fast | Computing a hash is quick, even for large inputs |
| Well spread | Similar inputs give very different hashes, so the values spread out evenly |
| Collisions possible | There are endless possible inputs but only a fixed number of outputs, so two different inputs can share a hash. A good function makes this rare |

**Behind the scenes:** let's build two toy hash functions for strings, to see what makes one "good".

In [1]:
def sum_hash(text):
    """A toy hash: add up the character codes of the text."""
    total = 0
    for char in text:
        total += ord(char)  # ord() gives a character's number, e.g. ord("a") == 97
    return total

print(sum_hash("cat"))  # 312
print(sum_hash("cat"))  # 312 again: deterministic
print(sum_hash("act"))  # 312: a collision! the same letters in any order give the same sum
print(sum_hash("dog"))  # 314: similar words get similar hashes, so they bunch together

312
312
312
314


The sum ignores the **order** of the characters, so every anagram collides. A better function multiplies the running total before adding each character, so each character's position changes the result. This "multiply by 31, then add" recipe is the one Java uses for its strings.

In [2]:
def polynomial_hash(text):
    """A better toy hash: each character's position changes the result."""
    h = 0
    for char in text:
        h = h * 31 + ord(char)  # multiply first, so earlier characters get "shifted"
    return h % (2 ** 32)        # keep the result to a fixed size (32 bits)

print(polynomial_hash("cat"))  # 98262
print(polynomial_hash("act"))  # 96402: different order, different hash
print(polynomial_hash("dog"))
print(polynomial_hash("a very long sentence still becomes one fixed-size number"))

98262
96402
99644
2223976517


---
## 2. Python's built-in `hash()`

**What it is:** Python's own hash function. `dict` and `set` call it on every key you store or look up.

**Syntax:** `hash(obj)`

**Returns:** an `int` (it fits in 64 bits on a 64-bit computer).

In [3]:
# Small integers hash to themselves
print(hash(42))
print(hash(0))

# Other immutable values are hashable too
print(hash(3.5))
print(hash((1, 2, 3)))
print(hash(None))

42
0
1152921504606846979
529344067295497451
4238894112


**Fun fact:** `hash(-1)` is `-2`. Inside CPython's C code, `-1` is the signal for "an error happened", so the hash of `-1` is moved to `-2`.

In [4]:
print(hash(-1), hash(-2))  # both -2: a real collision, and that's fine

-2 -2


**The golden rule:** if two values are equal (`a == b`), their hashes **must** be equal. That's why `1`, `1.0` and `True` all hash to `1`, and why a `dict` treats them as the **same key**.

The reverse is not true: two values with the same hash don't have to be equal. That's a collision.

In [5]:
print(1 == 1.0 == True)                # True: equal values...
print(hash(1), hash(1.0), hash(True))  # ...so equal hashes

# All three are the same key, so the dict ends up with ONE entry.
# It keeps the first key object (1) and the last value ("bool").
print({1: "int", 1.0: "float", True: "bool"})

True
1 1 1
{1: 'bool'}


**Hash randomization:** strings (and bytes) hash to a different number every time Python starts. This is a security feature. If attackers could predict string hashes, they could send a web server thousands of keys that all collide and make its dicts extremely slow (a *hash flooding* attack).

Within one running program, a string's hash stays the same. Each run below starts a brand-new Python process, so the hash of `"hello"` changes. This is why you should never save `hash()` values in a file or send them to another program.

In [6]:
import subprocess
import sys

print("this kernel:", hash("hello"), hash("hello"))  # stable within one process

# Start 3 separate Python processes; each one prints hash("hello")
for run in range(1, 4):
    result = subprocess.run(
        [sys.executable, "-c", "print(hash('hello'))"],  # sys.executable is the current Python
        capture_output=True,
        text=True,
    )
    print(f"new process {run}:", result.stdout.strip())

this kernel: -187163029737975278 -187163029737975278
new process 1: 5848984748413036649


new process 2: 809300247032619222


new process 3: 2588840351468780276


---
## 3. Hashable vs unhashable

**What it is:** a value is **hashable** if `hash()` works on it. Only hashable values can be `dict` keys or `set` items.

| Hashable (immutable) | Unhashable (mutable) |
|---|---|
| `int`, `float`, `bool`, `complex` | `list` |
| `str`, `bytes` | `dict` |
| `tuple` (if everything inside is hashable) | `set` |
| `frozenset`, `None` | |

**Behind the scenes, why mutable types are unhashable:** a dict stores each key in the bucket chosen by its hash. If you could change a list after using it as a key, its hash would change too, and the dict would look for it in the wrong bucket. The key would be lost inside the dict. Python prevents this by refusing to hash mutable types.

In [7]:
# Mutable types can't be hashed
for value in ([1, 2], {"a": 1}, {1, 2}):
    try:
        hash(value)
    except TypeError as error:
        print("TypeError:", error)

TypeError: unhashable type: 'list'
TypeError: unhashable type: 'dict'
TypeError: unhashable type: 'set'


In [8]:
# A tuple is hashable only if everything inside it is hashable
print(hash((1, 2, (3, 4))))  # works: every item inside is immutable

try:
    hash((1, [2, 3]))  # the list inside makes the whole tuple unhashable
except TypeError as error:
    print("TypeError:", error)

3794340727080330424
TypeError: unhashable type: 'list'


In [9]:
# Practical fix: convert a list to a tuple (or a set to a frozenset) to use it as a key
visited = {}
route = ["Pune", "Mumbai"]

try:
    visited[route] = True
except TypeError as error:
    print("TypeError:", error)

visited[tuple(route)] = True  # a tuple is hashable
print(visited)

tags = frozenset({"python", "hash"})  # an immutable version of a set
print({tags: "ok"})

TypeError: cannot use 'list' as a dict key (unhashable type: 'list')
{('Pune', 'Mumbai'): True}
{frozenset({'hash', 'python'}): 'ok'}


**Your own classes** are hashable by default (each object hashes by its identity). If you define what "equal" means with `__eq__`, you must also define `__hash__`, so that equal objects get equal hashes. Classes are covered in Session 06; for now, just note the rule.

In [10]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):  # two points are equal if their coordinates match
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):  # equal points -> equal tuples -> equal hashes
        return hash((self.x, self.y))


locations = {Point(1, 2): "home"}
print(locations[Point(1, 2)])  # a different but equal Point finds the same entry

home


---
## 4. How a hash table works

**What it is:** a **hash table** is a list of slots called **buckets**. Python's `dict` and `set` are both hash tables.

**Behind the scenes, storing `phone["asha"] = "98450-11111"`:**

1. Compute `hash("asha")`, which is a big integer.
2. Turn it into a bucket number with `hash("asha") % number_of_buckets`. The `%` (remainder) always gives a number from `0` to `number_of_buckets - 1`.
3. Store the pair `("asha", "98450-11111")` in that bucket.

**Looking up `phone["asha"]`** repeats steps 1 and 2 to find the bucket, then checks the key stored there. It never scans the other buckets, which is why lookups are fast.

```
key "asha" ──hash()──▶ 7041952983422874105 ──% 8──▶ bucket 1

 bucket 0: []
 bucket 1: [("asha", "98450-11111")]   ◀── stored here
 bucket 2: []
   ...
 bucket 7: []
```

(The hash number above is only an example: string hashes change on every run.)

In [11]:
buckets = 8

# Small integers hash to themselves, so their buckets are easy to predict
for key in [3, 10, 16, 25]:
    print(f"{key:>3} -> hash {hash(key):>3} -> bucket {hash(key) % buckets}")

  3 -> hash   3 -> bucket 3
 10 -> hash  10 -> bucket 2
 16 -> hash  16 -> bucket 0
 25 -> hash  25 -> bucket 1


In [12]:
# String hashes are huge (and change every run), but % always maps them into 0..7
for key in ["asha", "ravi", "monal", "alex"]:
    print(f"{key:<6} -> bucket {hash(key) % buckets}")

asha   -> bucket 3
ravi   -> bucket 6
monal  -> bucket 5
alex   -> bucket 6


Let's build that with plain lists and two functions, before turning it into a class in section 7. Each bucket is a list of `(key, value)` pairs.

In [13]:
NUM_BUCKETS = 8
# 8 separate empty lists. Not [[]] * 8, which would make 8 references to ONE list
table = [[] for _ in range(NUM_BUCKETS)]


def put(table, key, value):
    """Store value under key in the key's bucket."""
    bucket = table[hash(key) % len(table)]
    for i, (existing_key, _) in enumerate(bucket):
        if existing_key == key:
            bucket[i] = (key, value)  # the key is already there: update its value
            return
    bucket.append((key, value))       # a new key: add it to the bucket


def lookup(table, key):
    """Return the value stored under key, or None if it's missing."""
    bucket = table[hash(key) % len(table)]
    for existing_key, value in bucket:  # only this one bucket is searched
        if existing_key == key:
            return value
    return None


put(table, 3, "three")
put(table, 10, "ten")
put(table, 16, "sixteen")
put(table, 10, "TEN")  # the same key again: updates instead of adding a second entry

for index, bucket in enumerate(table):
    print(f"bucket {index}: {bucket}")

print(lookup(table, 10))  # TEN
print(lookup(table, 99))  # None: 99 % 8 is 3, and bucket 3 only holds key 3

bucket 0: [(16, 'sixteen')]
bucket 1: []
bucket 2: [(10, 'TEN')]
bucket 3: [(3, 'three')]
bucket 4: []
bucket 5: []
bucket 6: []
bucket 7: []
TEN
None


---
## 5. Collisions

**What it is:** a **collision** happens when two different keys land in the same bucket. When there are more possible keys than buckets, collisions can't be avoided. This is the *pigeonhole principle*: put 9 pigeons into 8 holes and at least one hole gets two. So every hash table needs a way to handle them.

| Strategy | How it works | Used by |
|---|---|---|
| **Separate chaining** | Each bucket holds a list, and colliding keys go into the same list | Java's `HashMap`, and our `solution.py` |
| **Open addressing** | Each bucket holds one item. On a collision, probe other buckets until a free one is found | CPython's `dict` and `set` |

In [14]:
# Chaining: 1, 9 and 17 all leave remainder 1 when divided by 8, so they share bucket 1
table = [[] for _ in range(8)]
for key in (1, 9, 17, 4):
    put(table, key, f"value {key}")

for index, bucket in enumerate(table):
    print(f"bucket {index}: {bucket}")

# The lookup goes to bucket 1, then checks each pair in that bucket's list
print(lookup(table, 17))

bucket 0: []
bucket 1: [(1, 'value 1'), (9, 'value 9'), (17, 'value 17')]
bucket 2: []
bucket 3: []
bucket 4: [(4, 'value 4')]
bucket 5: []
bucket 6: []
bucket 7: []
value 17


**Open addressing** with *linear probing*: if a key's bucket is taken, try the next one, wrapping around to the start at the end. CPython uses a cleverer probe order, but the idea is the same.

In [15]:
def probe_insert(slots, key):
    """Put key into the first free slot, starting from its hash bucket.

    This sketch assumes there is always a free slot; a real table resizes first.
    """
    index = hash(key) % len(slots)
    while slots[index] is not None:       # slot taken: a collision
        index = (index + 1) % len(slots)  # try the next slot, wrapping around to 0
    slots[index] = key
    return index


slots = [None] * 8  # one key per slot ([None] * 8 is fine: None is immutable)
for key in (1, 9, 17, 4):
    print(f"key {key:>2} -> slot {probe_insert(slots, key)}")
print(slots)

key  1 -> slot 1
key  9 -> slot 2
key 17 -> slot 3
key  4 -> slot 4
[None, 1, 9, 17, 4, None, None, None]


---
## 6. Load factor and resizing

**What it is:** the **load factor** is `number of items / number of buckets`. As it grows, collisions get more common, buckets get longer, and lookups slow down.

**Behind the scenes:** when the load factor passes a limit, the table **resizes**. It creates more buckets (usually twice as many) and moves every item, because `hash(key) % new_size` gives each key a new position. A resize is slow, but it only happens each time the size doubles, so on average adding an item stays fast.

| Table | Resizes when the load factor passes |
|---|---|
| Our `HashTable` | 0.75 |
| Java `HashMap` | 0.75 |
| CPython `dict` | about 2/3 |

You can watch a real `dict` resize: its memory size jumps at certain points instead of growing on every insert. The exact numbers depend on your Python version.

In [16]:
import sys

d = {}
last_size = sys.getsizeof(d)  # memory used by the dict itself, in bytes
print(f"0 items: {last_size} bytes")

for n in range(1, 50):
    d[n] = n
    size = sys.getsizeof(d)
    if size != last_size:  # print only when the dict grew its internal table
        print(f"{n} items: {size} bytes  <- resized")
        last_size = size

0 items: 64 bytes
1 items: 224 bytes  <- resized
6 items: 352 bytes  <- resized
11 items: 632 bytes  <- resized
22 items: 1168 bytes  <- resized
43 items: 2264 bytes  <- resized


---
## 7. Building our own hash table (`solution.py`)

[solution.py](solution.py) puts everything above into one class, `HashTable`:

| Method | What it does |
|---|---|
| `set(key, value)` or `table[key] = value` | Add a key, or update it if it already exists |
| `get(key)` or `table[key]` | Return the value; raise `KeyError` if the key is missing |
| `delete(key)` or `del table[key]` | Remove the key; raise `KeyError` if it is missing |
| `key in table`, `len(table)` | Membership test, and the number of items |
| `items()`, `keys()` | All `(key, value)` pairs, or all keys |
| `bucket_index(key)` | Which bucket a key belongs in |
| `show()` | Print every bucket, to see collisions |
| `capacity`, `load_factor` | The number of buckets, and how full they are |

It uses separate chaining, and doubles its buckets when the load factor passes 0.75. The tests in [test_solution.py](test_solution.py) check every method, and compare the table with a real `dict` after hundreds of operations.

In [17]:
from solution import HashTable  # import our class from solution.py in this folder

table = HashTable(capacity=4)
print(table)

HashTable({}, capacity=4)


In [18]:
# Watch the capacity double once the load factor goes above 0.75
for key in [1, 2, 3, 4, 5]:
    table.set(key, key * 100)
    print(f"added {key}: size={len(table)}, capacity={table.capacity}, load factor={table.load_factor:.2f}")

added 1: size=1, capacity=4, load factor=0.25
added 2: size=2, capacity=4, load factor=0.50
added 3: size=3, capacity=4, load factor=0.75
added 4: size=4, capacity=8, load factor=0.50
added 5: size=5, capacity=8, load factor=0.62


In [19]:
table.show()  # after the resize to 8 buckets, keys 1 to 5 each have their own bucket

bucket 0: []
bucket 1: [(1, 100)]
bucket 2: [(2, 200)]
bucket 3: [(3, 300)]
bucket 4: [(4, 400)]
bucket 5: [(5, 500)]
bucket 6: []
bucket 7: []


In [20]:
# Lookup, update, delete and membership, with dict-style syntax
print(table.get(3))         # 300
table[3] = "three hundred"  # the same key: its value is replaced
print(table[3])

del table[3]
print(3 in table)  # False
print(len(table))  # 4

try:
    table.get(3)
except KeyError as error:
    print("KeyError:", error)

300
three hundred
False
4
KeyError: 3


In [21]:
# Collisions with chaining: 1, 9 and 17 share bucket 1.
# A high max_load_factor stops the table from resizing, so the chain stays visible.
crowded = HashTable(capacity=8, max_load_factor=10)
for key in (1, 9, 17, 4):
    crowded.set(key, f"value {key}")

crowded.show()
print(crowded.get(17))

bucket 0: []
bucket 1: [(1, 'value 1'), (9, 'value 9'), (17, 'value 17')]
bucket 2: []
bucket 3: []
bucket 4: [(4, 'value 4')]
bucket 5: []
bucket 6: []
bucket 7: []
value 17


In [22]:
# String keys work too (their buckets change every run, because of hash randomization)
contacts = HashTable()
contacts["asha"] = "98450-11111"
contacts["ravi"] = "98450-22222"
contacts["monal"] = "98450-33333"

print(contacts)
print(contacts["ravi"])
print("monal" in contacts)

HashTable({'asha': '98450-11111', 'monal': '98450-33333', 'ravi': '98450-22222'}, capacity=8)
98450-22222
True


---
## 8. Why lookups are fast

| Operation | `list` | `dict` / `set` (on average) |
|---|---|---|
| `x in collection` | O(n): checks the items one by one | O(1): hash → bucket → done |
| Find a value by its key | O(n) | O(1) |
| Add an item | O(1) at the end | O(1) (an occasional resize is slower, but rare) |

**O(1)**, or "constant time", means the time doesn't grow as the collection grows. **O(n)** means the time grows in proportion to the number of items. In the worst case, when every key collides, a hash table slows down to O(n). That's why a good hash function and resizing matter.

Timings depend on your computer, but the gap is always huge:

In [23]:
import timeit

n = 100_000
numbers_list = list(range(n))
numbers_set = set(numbers_list)
numbers_dict = dict.fromkeys(numbers_list)  # {0: None, 1: None, ...}

target = n - 1  # the last item: the worst case for a list, which scans from the front

# Time 1,000 membership checks on each collection
list_time = timeit.timeit(lambda: target in numbers_list, number=1_000)
set_time = timeit.timeit(lambda: target in numbers_set, number=1_000)
dict_time = timeit.timeit(lambda: target in numbers_dict, number=1_000)

print(f"list: {list_time:.4f} s")
print(f"set:  {set_time:.6f} s")
print(f"dict: {dict_time:.6f} s")
print(f"the set was roughly {list_time / set_time:,.0f} times faster than the list")

list: 1.1724 s
set:  0.000049 s
dict: 0.000053 s
the set was roughly 24,007 times faster than the list


---
## 9. Cryptographic hashes with `hashlib`

`hash()` is built for speed inside one running program. When a hash has to be secure, or has to stay the same everywhere (in files, on other computers, years later), use the `hashlib` module. It provides **cryptographic hash functions** such as SHA-256.

| | `hash()` | `hashlib.sha256()` |
|---|---|---|
| Purpose | Fast `dict` / `set` lookups | Security, integrity checks, IDs |
| Output | An `int` (64 bits) | 256 bits, usually shown as 64 hex characters |
| Same value on every run | No, for strings (randomized) | Yes, always |
| Practically collision-free | No | Yes |
| Can you get the input back? | Not the point | No: it's designed to be one-way |

**Syntax:** `hashlib.sha256(data).hexdigest()`. The input must be **bytes**, so call `.encode()` on a string first.

**Returns:** the hash as a string of hexadecimal characters.

In [24]:
import hashlib

digest = hashlib.sha256("hello".encode()).hexdigest()
print(digest)       # identical on every run and on every computer
print(len(digest))  # 64 hex characters = 256 bits

2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
64


**The avalanche effect:** changing a single character changes the whole hash. That makes even a tiny change to a file or message easy to detect.

In [25]:
print(hashlib.sha256("hello".encode()).hexdigest())
print(hashlib.sha256("Hello".encode()).hexdigest())  # one capital letter: a completely different hash

2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969


**Where cryptographic hashes are used:**

- **File integrity:** download sites publish a file's SHA-256, so you can check that your copy wasn't corrupted or tampered with.
- **Git:** every commit, file and folder is identified by its hash. A commit ID like `990759e` in this repo is the start of one.
- **Password storage:** websites store a hash of your password, not the password itself. They add a random **salt** and use a deliberately slow algorithm, so leaked hashes are hard to crack.
- **Deduplication and caching:** identical content gives an identical hash, so duplicates are easy to spot.

> MD5 and SHA-1 are older algorithms with known collision attacks. Don't use them for anything security-related; use SHA-256 or newer.

In [26]:
def checksum(text):
    """Return the SHA-256 hash of some text."""
    return hashlib.sha256(text.encode()).hexdigest()


original = "Transfer 1,000 rupees to Asha"
published_checksum = checksum(original)  # sent or stored separately from the message

received = "Transfer 9,000 rupees to Asha"  # tampered with on the way
print("intact" if checksum(received) == published_checksum else "changed!")
print("intact" if checksum(original) == published_checksum else "changed!")

changed!
intact


In [27]:
import os

# Store a salted, slow hash instead of the password itself
password = "my-secret-password"
salt = os.urandom(16)  # 16 random bytes, different for every user
stored_hash = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 100_000)  # 100,000 rounds: slow on purpose

# At login: hash the attempt with the same salt, then compare.
# (Real code compares with hmac.compare_digest, or uses a library such as bcrypt or argon2.)
attempt = "my-secret-password"
attempt_hash = hashlib.pbkdf2_hmac("sha256", attempt.encode(), salt, 100_000)
print("login ok" if attempt_hash == stored_hash else "wrong password")

login ok


---
## 10. Practice exercises

Each exercise uses a `dict` or `set` (a hash table) to solve a common problem efficiently. Try each one yourself first, then compare with the solution.

**1. Word frequency.** Count how many times each word appears in a sentence.

In [28]:
# Solution 1: word frequency
sentence = "the cat sat on the mat the end"

counts = {}
for word in sentence.split():
    counts[word] = counts.get(word, 0) + 1  # get() returns 0 the first time a word is seen

print(counts)

{'the': 3, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1, 'end': 1}


Each `counts[word]` lookup is O(1), so counting n words is O(n). With a list of `[word, count]` pairs, you would have to scan the list for every word.

**2. Find duplicates.** Return the values that appear more than once in a list.

In [29]:
# Solution 2: find duplicates with two sets
def find_duplicates(values):
    seen = set()
    duplicates = set()
    for value in values:
        if value in seen:  # O(1) membership check
            duplicates.add(value)
        else:
            seen.add(value)
    return duplicates

print(find_duplicates([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]))  # {1, 3, 5}

{1, 3, 5}


**3. Two sum.** Given a list of numbers and a target, return the positions of two numbers that add up to the target. This is a classic interview question.

In [30]:
# Solution 3: two sum in one pass, with a dict of {number: index}
def two_sum(numbers, target):
    seen = {}  # the numbers passed so far -> their index
    for index, number in enumerate(numbers):
        needed = target - number
        if needed in seen:  # O(1): have we already passed this number's partner?
            return seen[needed], index
        seen[number] = index
    return None

print(two_sum([2, 7, 11, 15], 9))  # (0, 1): 2 + 7
print(two_sum([3, 2, 4], 6))       # (1, 2): 2 + 4
print(two_sum([1, 2, 3], 100))     # None

(0, 1)
(1, 2)
None


Checking every pair of numbers would be O(n²). The dict remembers what we've already seen, which makes it O(n).

**4. Group anagrams.** Group together the words that are made of the same letters. Hint: sorted letters make a good key, but `sorted()` returns a list, which isn't hashable.

In [31]:
# Solution 4: group anagrams, using a tuple of sorted letters as the key
def group_anagrams(words):
    groups = {}
    for word in words:
        key = tuple(sorted(word))  # "eat" -> ('a', 'e', 't'); a tuple, because a list can't be a key
        groups.setdefault(key, []).append(word)  # setdefault() creates an empty list the first time
    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


---
## Summary

- A **hash function** maps any input to a fixed-size number. The same input always gives the same hash; collisions are possible but should be rare.
- `hash(x)` is Python's built-in hash. If `a == b`, then `hash(a) == hash(b)`, which is why `1`, `1.0` and `True` are the same dict key.
- String hashes are **randomized on every run** for security, so never store `hash()` values.
- Only **hashable** (immutable) values can be dict keys or set items. Convert a list to a tuple first.
- A **hash table** finds a key's bucket with `hash(key) % number_of_buckets`, so a lookup goes straight to the right place: **O(1)** on average.
- **Collisions** are handled by chaining (a list per bucket) or by open addressing (probing for a free slot, which is what CPython does).
- When the **load factor** gets too high, the table **resizes** and moves every item to its new bucket.
- For security and for hashes that must stay the same everywhere, use **`hashlib`** (SHA-256), not `hash()`.